In [1]:
import torch
import torch.nn as nn

In [2]:
linear = nn.Linear(4, 3)
print(linear.weight.shape)

torch.Size([3, 4])


In [3]:
class LoRALinear(nn.Module):

    def __init__(self, in_features, out_features,rank=2, alpha=4):
        super().__init__()
        self.linear = nn.Linear(in_features,out_features)
        for param in self.linear.parameters():
            param.requires_grad = False

        self.A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank
    def forward(self, x):

        original_output = self.linear(x)
        lora_output = (x @ self.A.T @ self.B.T)

        return original_output + (self.scaling * lora_output)

In [4]:
model = LoRALinear(in_features=4,out_features=3,rank=2,alpha=4)

In [5]:
for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)

A torch.Size([2, 4]) True
B torch.Size([3, 2]) True
linear.weight torch.Size([3, 4]) False
linear.bias torch.Size([3]) False


In [6]:
x = torch.randn(5, 4)
output = model(x)
print(output.shape)

torch.Size([5, 3])


In [7]:
target = torch.randn(5, 3)
criterion = nn.MSELoss()
loss = criterion(output, target)
loss.backward()

In [8]:
print(model.A.grad is not None)
print(model.B.grad is not None)
print(model.linear.weight.grad)

True
True
None
